# 0 IMPORTACIÓN DE BIBLIOTECAS Y LIBRERÍAS

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import classification_report
import numpy as np
import joblib

# 1 INGESTIÓN O CARGA DE DATOS

In [2]:
df_original = pd.read_csv(r"https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv")

print(f"TAMAÑO DEL DATASET: {df_original.shape[0]} filas x {df_original.shape[1]} columnas")

TAMAÑO DEL DATASET: 891 filas x 3 columnas


# 2 INSPECCIÓN INICIAL

## 2.1 VISUALIZACIÓN DE TODAS LAS COLUMNAS 

In [3]:
pd.set_option('display.max_columns', None)
# Esto sirve para que me muestre todas las columnas cuando un DataSet es muy ancho o con muchas columnas 
# Con el (None) le digo que me muestre todas las columnas sin limitación
df_original.head(3)

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0


## 2.2 INFORMACIÓN DEL DATASET 

In [4]:
# Información general del dataset
print("INFORMACIÓN GENERAL:")
print("="*80)
df_original.info()

print("\n" + "="*80)

# Lista de columnas
print("COLUMNAS DEL DATASET:\n")
print(df_original.columns.tolist())

INFORMACIÓN GENERAL:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   package_name  891 non-null    object
 1   review        891 non-null    object
 2   polarity      891 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 21.0+ KB

COLUMNAS DEL DATASET:

['package_name', 'review', 'polarity']


# 3 CRIBADO DE VARIABLES 

## 3.1 DFINICIÓN DE OBJETIVO DE ESTUDIO 

En este caso, tenemos solo 3 variables: 2 predictoras y una etiqueta dicotómica. De las dos predictoras, realmente solo nos interesa la parte del comentario, ya que el hecho de clasificar un comentario en positivo o negativo dependerá de su contenido, no de la aplicación de la que se haya escrito. Por lo tanto, la variable package_name habría que eliminarla.

Cuando trabajamos con textos como en este caso, no tiene sentido hacer un EDA, el proceso es diferente, ya que la única variable que nos interesa es la que contiene el texto. En otros casos en los que el texto formase parte de un conjunto complejo con otras variables predictoras numéricas y el objetivo de predicción sea distinto, entonces tiene sentido aplicar un EDA.

-package_name. Nombre de la aplicación móvil (categórico)

-review. Comentario sobre la aplicación móvil (categórico)

-polarity. Variable de clase (0 o 1), siendo 0 un comentario negativo y 1, positivo (categórico numérico)

## 3.2 CREACIÓN DE DATASET PARA TRABAJAR

In [5]:
df = df_original.copy()

## 3.2 ELIMINAR LA VARIABLE PACKAGE_NAME

In [6]:
df = df.drop(columns=['package_name'])

## 3.3 ELIMINAR ESPACIOS Y CONVERTIR A MINÚSCULAS EL TEXTO 

In [7]:
df["review"] = df["review"].str.strip().str.lower()

# 4 DIVISIÓN DEL DATASET TRAIN-TEST

In [8]:
# Separar variables predictoras (X) y objetivo (y)
review = df['review']
polarity = df['polarity']

print(f"Total de comentarios: {len(review)}")
print(f"Distribución de polaridad:\n{polarity.value_counts()}")

Total de comentarios: 891
Distribución de polaridad:
polarity
0    584
1    307
Name: count, dtype: int64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(review, polarity, test_size=0.2, random_state=42)

print(f"Tamaño del conjunto de prueba: {len(X_test)}")
print(f"Tamaño del conjunto de entrenamiento: {len(X_train)}")

Tamaño del conjunto de prueba: 179
Tamaño del conjunto de entrenamiento: 712


# 5 VECTORIZAR 

In [10]:
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 6 MODELO MULTINOMIALNB

In [11]:
bayes = MultinomialNB().fit(X_train_vec, y_train)
y_pred = bayes.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.95      0.90       126
           1       0.84      0.58      0.69        53

    accuracy                           0.84       179
   macro avg       0.84      0.77      0.79       179
weighted avg       0.84      0.84      0.83       179



# 7 MODELO GAUSSIANNB

In [12]:
# Convertir a matriz densa con toarray() ESTO PUEDE CONSUMIR MUCHA MEMORIA
bayes = GaussianNB().fit(X_train_vec.toarray(), y_train)
y_pred = bayes.predict(X_test_vec.toarray())
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86       126
           1       0.70      0.60      0.65        53

    accuracy                           0.80       179
   macro avg       0.77      0.75      0.76       179
weighted avg       0.80      0.80      0.80       179



# 8 MODELO BERNOULLINB

In [13]:
bayes = BernoulliNB().fit(X_train_vec, y_train)
y_pred = bayes.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.94      0.90       126
           1       0.81      0.64      0.72        53

    accuracy                           0.85       179
   macro avg       0.84      0.79      0.81       179
weighted avg       0.85      0.85      0.84       179



CONCLUSIÓN: EL MEJOR MODELO ES EL DE BERNOULLINB

# 9 OPTIMIZACIÓN DEL MODELO DE BERNOULLINB

## 9.1 BÚSQUEDA EXTENSIVA CON GRIDSEARCHCV

In [14]:
# Definir el espacio de hiperparámetros para BernoulliNB
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],  # Suavizado de Laplace
    'binarize': [0.0, 0.5, 1.0, 2.0, None],  # Umbral para binarizar características
    'fit_prior': [True, False]  # Si usar probabilidades a priori o uniformes
}

print("PARÁMETROS A PROBAR:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
    
print(f"\nTotal de combinaciones a probar: {total_combinations}")

PARÁMETROS A PROBAR:
  alpha: [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
  binarize: [0.0, 0.5, 1.0, 2.0, None]
  fit_prior: [True, False]

Total de combinaciones a probar: 80


In [15]:
# Crear el modelo BernoulliNB
model_bernoulli = BernoulliNB()

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=model_bernoulli,
    param_grid=param_grid,
    cv=5,  # Validación cruzada de 5 folds
    scoring='accuracy',  # Métrica de evaluación
    n_jobs=-1,  # Usar todos los procesadores disponibles
    verbose=2  # Mostrar progreso
)
# Ajustar el modelo
grid_search.fit(X_train_vec, y_train)

Fitting 5 folds for each of 80 candidates, totalling 400 fits


,estimator,BernoulliNB()
,param_grid,"{'alpha': [0.001, 0.01, ...], 'binarize': [0.0, 0.5, ...], 'fit_prior': [True, False]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.0


In [16]:
# Evaluar el mejor modelo con classification_report
best_model_grid = grid_search.best_estimator_
y_pred_grid = best_model_grid.predict(X_test_vec)

print("REPORTE DE CLASIFICACIÓN - MODELO OPTIMIZADO CON GRIDSEARCHCV")
print("="*80)
print(classification_report(y_test, y_pred_grid))

REPORTE DE CLASIFICACIÓN - MODELO OPTIMIZADO CON GRIDSEARCHCV
              precision    recall  f1-score   support

           0       0.87      0.93      0.90       126
           1       0.80      0.68      0.73        53

    accuracy                           0.85       179
   macro avg       0.84      0.80      0.82       179
weighted avg       0.85      0.85      0.85       179



## 9.2 BÚSQUEDA INTENSIVA CON RANDOMIZEDSEARCHCV

In [17]:
# Definir distribuciones de hiperparámetros (más amplias que GridSearch)
param_distributions = {
    'alpha': np.logspace(-3, 2, 100),  # 100 valores entre 0.001 y 100
    'binarize': [None] + list(np.linspace(0, 5, 20)),  # None + 20 valores entre 0 y 5
    'fit_prior': [True, False]
}

print("DISTRIBUCIONES DE PARÁMETROS PARA BÚSQUEDA ALEATORIA:")
print(f"  alpha: 100 valores logarítmicos entre 0.001 y 100")
print(f"  binarize: None + 20 valores entre 0 y 5")
print(f"  fit_prior: [True, False]")
print(f"\nSe probarán 50 combinaciones aleatorias")

DISTRIBUCIONES DE PARÁMETROS PARA BÚSQUEDA ALEATORIA:
  alpha: 100 valores logarítmicos entre 0.001 y 100
  binarize: None + 20 valores entre 0 y 5
  fit_prior: [True, False]

Se probarán 50 combinaciones aleatorias


In [18]:
# Configurar RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=BernoulliNB(),
    param_distributions=param_distributions,
    n_iter=50,  # Número de combinaciones a probar
    cv=5,  # Validación cruzada de 5 folds
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    random_state=42  # Para reproducibilidad
)

# Ajustar el modelo
random_search.fit(X_train_vec, y_train)


Fitting 5 folds for each of 50 candidates, totalling 250 fits


,estimator,BernoulliNB()
,param_distributions,"{'alpha': array([1.0000...00000000e+02]), 'binarize': [None, np.float64(0.0), ...], 'fit_prior': [True, False]}"
,n_iter,50
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [19]:
# Evaluar el mejor modelo con classification_report
best_model_random = random_search.best_estimator_
y_pred_random = best_model_random.predict(X_test_vec)

print("REPORTE DE CLASIFICACIÓN - MODELO OPTIMIZADO CON RANDOMIZEDSEARCHCV")
print("="*80)
print(classification_report(y_test, y_pred_random))

REPORTE DE CLASIFICACIÓN - MODELO OPTIMIZADO CON RANDOMIZEDSEARCHCV
              precision    recall  f1-score   support

           0       0.88      0.91      0.90       126
           1       0.78      0.72      0.75        53

    accuracy                           0.85       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.85      0.85      0.85       179



## 9.3 COMPARACIÓN FINAL DE MODELOS

In [20]:
# Comparar todos los modelos
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

print("COMPARACIÓN FINAL DE MODELOS")
print("="*80)

# Modelo base (sin optimización)
model_base = BernoulliNB()
model_base.fit(X_train_vec, y_train)
y_pred_base = model_base.predict(X_test_vec)

# Calcular métricas para todos los modelos
def get_metrics(y_true, y_pred):
    precision, recall, fscore, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    precision_macro, recall_macro, fscore_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)
    return {
        'accuracy': accuracy,
        'precision_class_0': precision[0],
        'recall_class_0': recall[0],
        'f1_class_0': fscore[0],
        'precision_class_1': precision[1],
        'recall_class_1': recall[1],
        'f1_class_1': fscore[1],
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': fscore_macro
    }

metrics_base = get_metrics(y_test, y_pred_base)
metrics_grid = get_metrics(y_test, y_pred_grid)
metrics_random = get_metrics(y_test, y_pred_random)

# Crear DataFrame comparativo
comparison_df = pd.DataFrame({
    'BernoulliNB Base': metrics_base,
    'GridSearchCV': metrics_grid,
    'RandomizedSearchCV': metrics_random
})

print("\nTABLA COMPARATIVA DE MÉTRICAS:")
print(comparison_df.round(4))

print("\n" + "="*80)
print("\nDETALLE POR MODELO:")
print("\n1. BernoulliNB base (sin optimización):")
print(f"   Accuracy: {metrics_base['accuracy']:.4f}")
print(f"   F1-Score Macro: {metrics_base['f1_macro']:.4f}")

print(f"\n2. BernoulliNB optimizado con GridSearchCV:")
print(f"   Hiperparámetros: {grid_search.best_params_}")
print(f"   Accuracy: {metrics_grid['accuracy']:.4f}")
print(f"   F1-Score Macro: {metrics_grid['f1_macro']:.4f}")
print(f"   Mejora en Accuracy: {(metrics_grid['accuracy'] - metrics_base['accuracy'])*100:+.2f}%")
print(f"   Mejora en F1-Macro: {(metrics_grid['f1_macro'] - metrics_base['f1_macro'])*100:+.2f}%")

print(f"\n3. BernoulliNB optimizado con RandomizedSearchCV:")
print(f"   Hiperparámetros: {random_search.best_params_}")
print(f"   Accuracy: {metrics_random['accuracy']:.4f}")
print(f"   F1-Score Macro: {metrics_random['f1_macro']:.4f}")
print(f"   Mejora en Accuracy: {(metrics_random['accuracy'] - metrics_base['accuracy'])*100:+.2f}%")
print(f"   Mejora en F1-Macro: {(metrics_random['f1_macro'] - metrics_base['f1_macro'])*100:+.2f}%")

print("\n" + "="*80)
print("\nANÁLISIS POR CLASE:")
print(f"\nClase 0 (Comentarios Negativos):")
print(f"  Mejor Precision: {'GridSearchCV' if metrics_grid['precision_class_0'] >= metrics_random['precision_class_0'] else 'RandomizedSearchCV'} ({max(metrics_grid['precision_class_0'], metrics_random['precision_class_0']):.4f})")
print(f"  Mejor Recall: {'GridSearchCV' if metrics_grid['recall_class_0'] >= metrics_random['recall_class_0'] else 'RandomizedSearchCV'} ({max(metrics_grid['recall_class_0'], metrics_random['recall_class_0']):.4f})")
print(f"  Mejor F1-Score: {'GridSearchCV' if metrics_grid['f1_class_0'] >= metrics_random['f1_class_0'] else 'RandomizedSearchCV'} ({max(metrics_grid['f1_class_0'], metrics_random['f1_class_0']):.4f})")

print(f"\nClase 1 (Comentarios Positivos):")
print(f"  Mejor Precision: {'GridSearchCV' if metrics_grid['precision_class_1'] >= metrics_random['precision_class_1'] else 'RandomizedSearchCV'} ({max(metrics_grid['precision_class_1'], metrics_random['precision_class_1']):.4f})")
print(f"  Mejor Recall: {'GridSearchCV' if metrics_grid['recall_class_1'] >= metrics_random['recall_class_1'] else 'RandomizedSearchCV'} ({max(metrics_grid['recall_class_1'], metrics_random['recall_class_1']):.4f})")
print(f"  Mejor F1-Score: {'GridSearchCV' if metrics_grid['f1_class_1'] >= metrics_random['f1_class_1'] else 'RandomizedSearchCV'} ({max(metrics_grid['f1_class_1'], metrics_random['f1_class_1']):.4f})")

print("\n" + "="*80)
# Determinar el ganador basado en F1-Score macro (métrica más balanceada)
if metrics_grid['f1_macro'] > metrics_random['f1_macro']:
    print("🏆 GANADOR: GridSearchCV (mejor F1-Score macro)")
    print(f"   GridSearchCV es {((metrics_grid['f1_macro'] - metrics_random['f1_macro'])*100):.2f}% mejor")
elif metrics_random['f1_macro'] > metrics_grid['f1_macro']:
    print("🏆 GANADOR: RandomizedSearchCV (mejor F1-Score macro)")
    print(f"   RandomizedSearchCV es {((metrics_random['f1_macro'] - metrics_grid['f1_macro'])*100):.2f}% mejor")
else:
    print("🏆 EMPATE: Ambos modelos tienen el mismo rendimiento")

COMPARACIÓN FINAL DE MODELOS

TABLA COMPARATIVA DE MÉTRICAS:
                   BernoulliNB Base  GridSearchCV  RandomizedSearchCV
accuracy                     0.8492        0.8547              0.8547
precision_class_0            0.8613        0.8731              0.8846
recall_class_0               0.9365        0.9286              0.9127
f1_class_0                   0.8973        0.9000              0.8984
precision_class_1            0.8095        0.8000              0.7755
recall_class_1               0.6415        0.6792              0.7170
f1_class_1                   0.7158        0.7347              0.7451
precision_macro              0.8354        0.8366              0.8301
recall_macro                 0.7890        0.8039              0.8148
f1_macro                     0.8066        0.8173              0.8218


DETALLE POR MODELO:

1. BernoulliNB base (sin optimización):
   Accuracy: 0.8492
   F1-Score Macro: 0.8066

2. BernoulliNB optimizado con GridSearchCV:
   Hiperparámet

# 10 SELECCIÓN DEL MEJOR MODELO Y EXPORTACIÓN DEL MODELO FINAL 

In [21]:
joblib.dump(best_model_random, 'predictive_model_of_comments.pkl')
print("✅ Modelo guardado exitosamente como 'predictive_model_of_comments.pkl'")

✅ Modelo guardado exitosamente como 'predictive_model_of_comments.pkl'


In [22]:
# Cargamos el modelo
modelo_final = joblib.load('predictive_model_of_comments.pkl')